# RazorGuard NLI — AgentPay-IR v2 fine-tune (Colab)

Base model pinned to `cross-encoder/nli-deberta-v3-base` @ revision
`6c749ce3425cd33b46d187e45b92bbf96ee12ec7`. Label map: 0=contradiction, 1=entailment, 2=neutral.

**The bundle contains train+val only** — no frozen test, no human gold, no untouched OOD.
Selection uses validation only. Run top-to-bottom on T4/L4.


In [ ]:
import json, hashlib, zipfile, io, os, random
import torch
assert torch.cuda.is_available(), 'GPU required (Runtime > Change runtime type)'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import files
up = files.upload()  # upload agentpay_ir_v2_colab_training_bundle.zip
BUNDLE = next(iter(up))
sha = hashlib.sha256(open(BUNDLE,'rb').read()).hexdigest()
print('bundle sha256:', sha)


In [ ]:
!pip -q install transformers==4.55.4 datasets==4.0.1 accelerate==1.10.1 scikit-learn==1.7.1
zf = zipfile.ZipFile(BUNDLE)
manifest = json.loads(zf.read('bundle_manifest.json'))
assert hashlib.sha256(open(BUNDLE,'rb').read()).hexdigest() == manifest['bundle_sha256'], 'bundle hash mismatch'
zf.extractall('bundle')
for name, want in manifest['files'].items():  # manifest itself excluded
    h = hashlib.sha256(open('bundle/'+name,'rb').read()).hexdigest()
    assert h == want, f'hash mismatch {name}'
print('bundle verified:', manifest['files'])


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
REV = manifest['base_model_revision']
tok = AutoTokenizer.from_pretrained(manifest['base_model'], revision=REV)
print('label map:', manifest['label_map'])
set_seed(42)


In [ ]:
import torch
from torch.utils.data import Dataset
class NLIDataset(Dataset):
    def __init__(self, path, tok, max_len=256):
        self.rows = [json.loads(l) for l in open(path)]
        self.tok, self.max_len = tok, max_len
        self.lab = {'contradiction':0,'entailment':1,'neutral':2}
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        enc = self.tok(r['premise'], r['hypothesis'], truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        return {**{k: v[0] for k, v in enc.items()}, 'labels': torch.tensor(self.lab[r['label']])}
train_ds = NLIDataset('bundle/train.jsonl', tok)
val_ds = NLIDataset('bundle/val.jsonl', tok)
print('train', len(train_ds), 'val', len(val_ds))


In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
def metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    unsafe = int(((labels==0)&(preds==1)).sum())  # gold C predicted E
    c_rec = float((preds[labels==0]==0).mean()) if (labels==0).any() else 0.0
    n_rec = float((preds[labels==2]==2).mean()) if (labels==2).any() else 0.0
    e_fp_block = int(((labels==1)&(preds==0)).sum())  # safe entailments hard-blocked
    return {'macro_f1': f1_score(labels, preds, average='macro'),
            'contradiction_recall': c_rec, 'unsafe_c_to_e': unsafe,
            'neutral_recall': n_rec, 'safe_false_block': e_fp_block}
def run(epochs):
    set_seed(42)
    model = AutoModelForSequenceClassification.from_pretrained(manifest['base_model'], revision=REV, num_labels=3)
    args = TrainingArguments(f'out_{epochs}', num_train_epochs=epochs, learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=32, warmup_ratio=0.06,
        fp16=True, logging_steps=200, eval_strategy='epoch', save_strategy='no', report_to=[])
    tr = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=metrics)
    tr.train()
    ev = tr.evaluate()
    del tr, model; torch.cuda.empty_cache()
    return ev
results = {'A_2ep': run(2), 'B_3ep': run(3)}
print(json.dumps(results, indent=1))
# FROZEN selection rule (train_config.json): min unsafe C->E, then max macro-F1, then max contradiction recall
best = min(results.items(), key=lambda kv: (kv[1]['eval_unsafe_c_to_e'], -kv[1]['eval_macro_f1'], -kv[1]['eval_contradiction_recall']))[0]
print('SELECTED (validation only, frozen rule):', best)


In [ ]:
set_seed(42)
epochs = 2 if best=='A_2ep' else 3
model = AutoModelForSequenceClassification.from_pretrained(manifest['base_model'], revision=REV, num_labels=3)
args = TrainingArguments('final', num_train_epochs=epochs, learning_rate=2e-5,
    per_device_train_batch_size=16, warmup_ratio=0.06, fp16=True, logging_steps=200, report_to=[])
Trainer(model=model, args=args, train_dataset=train_ds).train()
model.save_pretrained('agentpay-ir-v2-finetuned', safe_serialization=True)
tok.save_pretrained('agentpay-ir-v2-finetuned')
json.dump(manifest['label_map'], open('agentpay-ir-v2-finetuned/label_map.json','w'))
open('agentpay-ir-v2-finetuned/base_model.txt','w').write(manifest['base_model'])
open('agentpay-ir-v2-finetuned/base_model_revision.txt','w').write(REV)
json.dump({'validation_results': results, 'selected': best, 'seed': 42}, open('agentpay-ir-v2-finetuned/training_metrics.json','w'), indent=1)
json.dump(manifest, open('agentpay-ir-v2-finetuned/dataset_manifest.json','w'), indent=1)
import transformers, accelerate, platform
def _sha(p):
    import hashlib
    return hashlib.sha256(open(p,'rb').read()).hexdigest()
artifact_files = {f: _sha('agentpay-ir-v2-finetuned/'+f) for f in os.listdir('agentpay-ir-v2-finetuned') if os.path.isfile('agentpay-ir-v2-finetuned/'+f)}
model_manifest = {
    'artifact': 'agentpay-ir-v2-finetuned',
    'base_model': manifest['base_model'],
    'base_model_revision': REV,
    'label_map': manifest['label_map'],
    'seed': 42,
    'selected_candidate': best,
    'candidate_results': results,
    'selection_rule': manifest and json.load(open('bundle/train_config.json'))['selection'],
    'dataset_manifest_files': manifest['files'],
    'dependency_versions': {'transformers': transformers.__version__, 'torch': torch.__version__, 'accelerate': accelerate.__version__, 'python': platform.python_version()},
    'artifact_files_sha256': artifact_files,
    'training_data_excluded': ['frozen test', 'human gold', 'untouched OOD'],
}
json.dump(model_manifest, open('agentpay-ir-v2-finetuned/model_manifest.json','w'), indent=1)
import shutil
shutil.make_archive('agentpay-ir-v2-finetuned', 'zip', '.', 'agentpay-ir-v2-finetuned')
files.download('agentpay-ir-v2-finetuned.zip')
print('DONE — place agentpay-ir-v2-finetuned.zip in artifacts/models/incoming/')
